# AICup 2026 VeriPromiseESG — Inference & Submission（A+MASK Multi-Seed 20-Model Ensemble）

用 A+MASK 訓練的 **4 個 seed × 5 fold = 20 個 checkpoints** 做 soft voting 推論，輸出 AICup 規範的 `submission.csv`。

## 本版改動點

- **4 組 5-fold checkpoint**（seed=1106910 + seed=1106 + seed=910 + seed=9910 → 共 20 個 model）
- A+MASK 結構（cascade + hierarchical mask）
- 模型架構含 **Cascading Task Heads（A）**：
  - `promise_status` 先預測 → softmax detach 後當 `timeline` / `evidence_status` 的額外特徵
  - `evidence_status` 預測 → softmax detach 後當 `evidence_quality` 的額外特徵
- 訓練時加了 **Hierarchical Loss Masking（MASK）**（推論不影響，純訓練機制）

## 流程

1. 解壓 zip → 5 個 EMA checkpoint
2. 自包含的 20 維特徵 + **cascade 版** MultiTaskRoberta 架構
3. **5 fold soft voting**（uniform 平均）+ 硬性邏輯約束
4. **標籤反向 normalize**：`longer_than_5_years` → `more_than_5_years`（AICup 規範）
5. 寫出 5 欄 CSV（UTF-8、id 順序 12001-14000、2000 筆）

## 重要相容性說明

Checkpoint 是 **20 維手工特徵 + cascade heads** 訓練的。task_heads 的 input 維度比舊版（平行 head）大：
- `promise_status`: 1088 維
- `verification_timeline / evidence_status`: 1088 + 2 = 1090 維
- `evidence_quality`: 1088 + 2 + 3 = 1093 維

Test 集**沒有 `esg_type`** 欄位（訓練集有），prefix 會自動退化成空字串 — 模型仍能運作但有 distribution shift 風險。


In [1]:
# === Step 1 — 路徑設定 + 解壓 2 個 A+MASK seed checkpoints（各 5 fold = 10 ckpt）===
import os
import zipfile

# Seed -> zip file 對應；OOF 階段必須用同一 seed 重建 K-fold split
CKPT_ZIPS = {
    1106910: "checkpoints_ema_AMASK_1106910.zip",
    1106:    "checkpoints_ema_AMASK_1106.zip",
    910:     "checkpoints_ema_AMASK_910.zip",
    9910:    "checkpoints_ema_AMASK_9910.zip",
}
CKPT_DIRS = {
    1106910: "checkpoints_inference_amask_1106910",
    1106:    "checkpoints_inference_amask_1106",
    910:     "checkpoints_inference_amask_910",
    9910:    "checkpoints_inference_amask_9910",
}
TEST_JSON  = "vpesg4k_test_2000.json"
OUTPUT_CSV = "submission.csv"

ckpt_paths_per_seed = {}     # seed -> [5 paths]
ckpt_paths          = []     # 全部 10 個（test inference 用）

for seed, zip_name in CKPT_ZIPS.items():
    ckpt_dir = CKPT_DIRS[seed]
    if not os.path.exists(ckpt_dir):
        with zipfile.ZipFile(zip_name) as z:
            z.extractall(ckpt_dir)
        print(f"✅ 解壓 {zip_name} → {ckpt_dir}/")
    else:
        print(f"ℹ️  {ckpt_dir}/ 已存在，跳過解壓")

    paths = sorted([
        os.path.join(ckpt_dir, f"fold_{i}", "best.pt") for i in range(1, 6)
    ])
    for p in paths:
        assert os.path.exists(p), f"找不到 {p}"
    ckpt_paths_per_seed[seed] = paths
    ckpt_paths.extend(paths)

print(f"\n📦 總共 {len(ckpt_paths)} 個 ckpt（{len(CKPT_ZIPS)} seeds × 5 folds）")
for p in ckpt_paths:
    print(f"  {p}  ({os.path.getsize(p)/1024/1024:.0f} MB)")


ℹ️  checkpoints_inference_amask_1106910/ 已存在，跳過解壓
ℹ️  checkpoints_inference_amask_1106/ 已存在，跳過解壓
ℹ️  checkpoints_inference_amask_910/ 已存在，跳過解壓
ℹ️  checkpoints_inference_amask_9910/ 已存在，跳過解壓

📦 總共 20 個 ckpt（4 seeds × 5 folds）
  checkpoints_inference_amask_1106910\fold_1\best.pt  (407 MB)
  checkpoints_inference_amask_1106910\fold_2\best.pt  (407 MB)
  checkpoints_inference_amask_1106910\fold_3\best.pt  (407 MB)
  checkpoints_inference_amask_1106910\fold_4\best.pt  (407 MB)
  checkpoints_inference_amask_1106910\fold_5\best.pt  (407 MB)
  checkpoints_inference_amask_1106\fold_1\best.pt  (407 MB)
  checkpoints_inference_amask_1106\fold_2\best.pt  (407 MB)
  checkpoints_inference_amask_1106\fold_3\best.pt  (407 MB)
  checkpoints_inference_amask_1106\fold_4\best.pt  (407 MB)
  checkpoints_inference_amask_1106\fold_5\best.pt  (407 MB)
  checkpoints_inference_amask_910\fold_1\best.pt  (407 MB)
  checkpoints_inference_amask_910\fold_2\best.pt  (407 MB)
  checkpoints_inference_amask_910\fold_3\b

In [2]:
# === Step 2 — Imports + Config + Label spaces ===
import json
import re
import math
import csv
from dataclasses import dataclass
from typing import Dict, List
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer
from tqdm.auto import tqdm


# 與訓練時 CFG 完全一致（A+MASK 版本：20 維特徵 + cascade heads）
@dataclass
class CFG:
    model_name: str = "hfl/chinese-roberta-wwm-ext"
    max_len: int = 512
    inference_batch_size: int = 16   # inference 不算 grad，可以放大
    # Hand features
    use_hand_features: bool = True
    num_hand_features: int = 20      # ⚠️ 訓練時是 20 維（14 raw + 5 derived + length）
    feature_proj_dim: int = 64
    # Span auxiliary
    use_span_pooling: bool = True
    span_pool_dim: int = 128
    use_cross_attention: bool = True
    cross_attn_dim: int = 256
    cross_attn_heads: int = 4
    use_task_xattn: bool = False
    num_span_labels: int = 5
    # ESG prefix
    use_esg_prefix: bool = True
    # ⚠️ A+MASK 新增：Cascading Task Heads
    # promise → 看 [combined]
    # timeline / evidence_status → 看 [combined, promise_probs]
    # evidence_quality → 看 [combined, promise_probs, evidence_probs]
    use_cascade_heads: bool = True
    # CORN-style Ordinal Regression for verification_timeline
    # ⚠️ 載入 ordinal-trained ckpt 需設 True；載入舊 ckpt 設 False（架構不同）
    use_ordinal_timeline: bool = False


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} | Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


EVAL_FIELDS = {
    "promise_status": ["Yes", "No"],
    "verification_timeline": [
        "already", "within_2_years", "between_2_and_5_years",
        "longer_than_5_years", "N/A",
    ],
    "evidence_status": ["Yes", "No", "N/A"],
    "evidence_quality": ["Clear", "Not Clear", "Misleading", "N/A"],
}
LABEL2ID = {f: {lab: i for i, lab in enumerate(labs)} for f, labs in EVAL_FIELDS.items()}
ID2LABEL = {f: {i: lab for i, lab in enumerate(labs)} for f, labs in EVAL_FIELDS.items()}
NUM_LABELS = {f: len(labs) for f, labs in EVAL_FIELDS.items()}


c:\Users\genji\Downloads\tf-safe-project\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch 2.6.0+cu124 | Device: cuda
GPU: NVIDIA GeForce RTX 3060 Laptop GPU


In [3]:
# === Step 3 — 20 維手工特徵（與 checkpoint 訓練時相同版本）===
# 14 raw patterns + 5 derived + 1 length = 20

PATTERNS = {
    "future_verbs":     re.compile(r"將|會|擬|計畫|預計|規劃|朝|致力於"),
    "percentages":      re.compile(r"\d+\.?\d*\s*%"),
    "years":            re.compile(r"20[2-4]\d\s*年"),
    "amounts":          re.compile(r"\d+\s*(?:億|萬|千萬)\s*元"),
    "vague":            re.compile(r"持續|積極|努力|適當|相關"),
    "goals":            re.compile(r"目標|KPI|指標"),
    "past_markers":     re.compile(r"已|目前|現行|迄今|截至|本期|累計|過去|自\s*20\d\d"),
    "evidence_verbs":   re.compile(r"推行|落實|導入|執行|建置|完成|達成|設置|設立|簽署|訂定|制定|通過|取得"),
    "numeric_specific": re.compile(r"\d+\.\d+|[1-9]\d{2,}"),
    "certification":    re.compile(r"ISO\s*\d+|TCFD|SASB|RE100|SBTi|永續會計準則"),
    "timeline_marker":  re.compile(r"於\s*20[2-4]\d\s*年(?:前|底|起|內)?|至\s*20[2-4]\d|預計於"),
    # 第 7 輪從 evidence_string chi-square mining 加進來的 3 個 pattern
    "strategic_vague":  re.compile(r"卓越|主軸|共創|中長期|碳策略|核心|藍圖|優化|期望"),
    "long_term_goal":   re.compile(r"2030|2050|2040|淨零|科學基礎|減量目標|減量路徑|脫碳|再生能源"),
    "near_term_action": re.compile(r"取得.{0,8}認證|簽署.{0,8}合約|供應商.{0,5}(?:配合|管理)|短期內"),
}
assert len(PATTERNS) == 14, f"應有 14 個 pattern，實際 {len(PATTERNS)}"


def extract_features(sample):
    text = sample.get("data", "") if isinstance(sample, dict) else sample
    counts = {name: len(p.findall(text)) for name, p in PATTERNS.items()}
    raw = [float(counts[name]) for name in PATTERNS.keys()]

    fw = counts["future_verbs"]
    pm = counts["past_markers"]
    future_ratio = fw / (fw + pm + 1.0)

    len_per_100 = max(len(text) / 100.0, 1.0)
    concrete = counts["percentages"] + counts["years"] + counts["amounts"]
    concrete_density = concrete / len_per_100
    vague_density = counts["vague"] / len_per_100
    goals_density = counts["goals"] / len_per_100
    evidence_verbs_density = counts["evidence_verbs"] / len_per_100

    vec = [math.log1p(x) for x in raw]
    vec += [
        future_ratio,
        min(concrete_density, 10.0),
        min(vague_density, 10.0),
        min(goals_density, 10.0),
        min(evidence_verbs_density, 10.0),
        math.log1p(len(text)),
    ]
    assert len(vec) == 20
    return vec


# Quick sanity check
_test = {"data": "本公司預計於 2030 年達成淨零排放目標，已通過 ISO 14064 認證，秉持卓越主軸推動中長期減碳路徑。"}
_feat = extract_features(_test)
print(f"特徵維度: {len(_feat)} (預期 20)")
print(f"sample 抓到的 pattern: " + ", ".join(
    [n for n, p in PATTERNS.items() if p.search(_test['data'])]
))


特徵維度: 20 (預期 20)
sample 抓到的 pattern: future_verbs, years, goals, past_markers, evidence_verbs, numeric_specific, certification, timeline_marker, strategic_vague, long_term_goal


In [4]:
# === Step 4 — 模型架構（嵌入式，與訓練時完全相同，支援 cascade heads） ===

class CrossAttentionBlock(nn.Module):
    def __init__(self, hidden_dim, attn_dim, num_heads, dropout=0.1):
        super().__init__()
        self.proj_down = nn.Linear(hidden_dim, attn_dim)
        self.attn = nn.MultiheadAttention(
            attn_dim, num_heads, dropout=dropout, batch_first=True,
        )
        self.norm1 = nn.LayerNorm(attn_dim)
        self.ff = nn.Sequential(
            nn.Linear(attn_dim, attn_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(attn_dim * 2, attn_dim),
        )
        self.norm2 = nn.LayerNorm(attn_dim)
        self.proj_up = nn.Linear(attn_dim, hidden_dim)

    def forward(self, x):
        proj = self.proj_down(x)
        attn_out, _ = self.attn(proj, proj, proj, need_weights=False)
        proj = self.norm1(proj + attn_out)
        ff_out = self.ff(proj)
        proj = self.norm2(proj + ff_out)
        out = self.proj_up(proj)
        return x + out


class OrdinalTimelineHead(nn.Module):
    """CORN-style ordinal regression head for verification_timeline.

    1 個 N/A binary + 3 個 CORN ordinal binary classifiers。
    Inference 時 to_5class_probs 把它轉成 [B, 5] 給下游用。
    """

    def __init__(self, in_dim, hidden, num_ord_classes=4):
        super().__init__()
        self.num_ord_classes = num_ord_classes
        self.shared = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.LayerNorm(hidden),
            nn.GELU(),
            nn.Dropout(0.3),
        )
        self.na_classifier = nn.Linear(hidden, 1)
        self.ordinal_classifier = nn.Linear(hidden, num_ord_classes - 1)

    def forward(self, x):
        h = self.shared(x)
        return self.na_classifier(h), self.ordinal_classifier(h)

    @staticmethod
    def to_5class_probs(na_logit, ordinal_logits, num_ord_classes=4):
        na_prob = torch.sigmoid(na_logit)
        ord_sig = torch.sigmoid(ordinal_logits)
        K = num_ord_classes
        B = na_logit.size(0)
        cumprod = torch.cumprod(ord_sig, dim=-1)
        p_ord = na_logit.new_zeros(B, K)
        p_ord[:, 0] = 1 - ord_sig[:, 0]
        for k in range(1, K - 1):
            p_ord[:, k] = cumprod[:, k - 1] * (1 - ord_sig[:, k])
        p_ord[:, K - 1] = cumprod[:, K - 2]
        non_na_prob = 1 - na_prob
        p_5class = na_logit.new_zeros(B, K + 1)
        p_5class[:, :K] = non_na_prob * p_ord
        p_5class[:, K] = na_prob.squeeze(-1)
        return p_5class


class AttentionPooler(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.attn = nn.Linear(hidden_size, 1)

    def forward(self, hidden_states, attention_mask):
        scores = self.attn(hidden_states).squeeze(-1)
        scores = scores.masked_fill(~attention_mask.bool(), -1e4)
        weights = scores.softmax(dim=-1).unsqueeze(-1)
        return (hidden_states * weights).sum(dim=1)


class MultiTaskRoberta(nn.Module):
    def __init__(self, num_labels_dict, model_name=None):
        super().__init__()
        model_name = model_name or CFG.model_name
        self.backbone = AutoModel.from_pretrained(model_name, output_hidden_states=True)
        hidden = self.backbone.config.hidden_size
        self.pooler = AttentionPooler(hidden)

        self.feature_proj = nn.Sequential(
            nn.Linear(CFG.num_hand_features, CFG.feature_proj_dim),
            nn.GELU(),
            nn.LayerNorm(CFG.feature_proj_dim),
        )

        if CFG.use_span_pooling:
            self.promise_proj = nn.Sequential(
                nn.Linear(hidden, CFG.span_pool_dim),
                nn.GELU(),
                nn.LayerNorm(CFG.span_pool_dim),
            )
            self.evidence_proj = nn.Sequential(
                nn.Linear(hidden, CFG.span_pool_dim),
                nn.GELU(),
                nn.LayerNorm(CFG.span_pool_dim),
            )
            if CFG.use_cross_attention:
                self.cross_attn = CrossAttentionBlock(
                    hidden_dim=hidden,
                    attn_dim=CFG.cross_attn_dim,
                    num_heads=CFG.cross_attn_heads,
                )

        head_input = hidden + CFG.feature_proj_dim
        if CFG.use_span_pooling:
            head_input += 2 * CFG.span_pool_dim

        # Cascade heads：下游 head input 含 upstream softmax 機率
        use_cascade = getattr(CFG, "use_cascade_heads", False)
        n_promise = num_labels_dict["promise_status"]
        n_evidence = num_labels_dict["evidence_status"]
        if use_cascade:
            head_inputs = {
                "promise_status":         head_input,
                "verification_timeline":  head_input + n_promise,
                "evidence_status":        head_input + n_promise,
                "evidence_quality":       head_input + n_promise + n_evidence,
            }
        else:
            head_inputs = {field: head_input for field in num_labels_dict}

        def _make_head(in_dim, out_dim):
            return nn.Sequential(
                nn.Linear(in_dim, hidden),
                nn.LayerNorm(hidden),
                nn.GELU(),
                nn.Dropout(0.3),
                nn.Linear(hidden, out_dim),
            )

        self.task_heads = nn.ModuleDict({
            field: _make_head(head_inputs[field], n)
            for field, n in num_labels_dict.items()
        })

        # CORN ordinal head 取代 verification_timeline 標準 head（如開啟）
        self.use_ordinal_timeline = getattr(CFG, "use_ordinal_timeline", False)
        if self.use_ordinal_timeline:
            n_timeline = num_labels_dict["verification_timeline"]
            self.task_heads["verification_timeline"] = OrdinalTimelineHead(
                in_dim=head_inputs["verification_timeline"],
                hidden=hidden,
                num_ord_classes=n_timeline - 1,
            )

        self.span_head = nn.Sequential(
            nn.Dropout(0.1),
            nn.Linear(hidden, CFG.num_span_labels),
        )

    def _span_weighted_pool(self, hidden_states, span_logits, attention_mask):
        span_probs = F.softmax(span_logits, dim=-1)
        promise_w = span_probs[..., 1] + span_probs[..., 2]
        evidence_w = span_probs[..., 3] + span_probs[..., 4]
        promise_w = promise_w * attention_mask
        evidence_w = evidence_w * attention_mask
        p_total = promise_w.sum(dim=1, keepdim=True).clamp(min=1e-6)
        e_total = evidence_w.sum(dim=1, keepdim=True).clamp(min=1e-6)
        promise_vec = (hidden_states * promise_w.unsqueeze(-1)).sum(dim=1) / p_total
        evidence_vec = (hidden_states * evidence_w.unsqueeze(-1)).sum(dim=1) / e_total
        return promise_vec, evidence_vec

    def forward(self, input_ids, attention_mask, features=None):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        last_4 = torch.stack(out.hidden_states[-4:], dim=0).mean(dim=0)
        span_logits = self.span_head(last_4)
        pooled = self.pooler(last_4, attention_mask)

        if features is None:
            features = pooled.new_zeros(pooled.size(0), CFG.num_hand_features)
        feat_emb = self.feature_proj(features)
        parts = [pooled, feat_emb]

        if CFG.use_span_pooling:
            promise_vec, evidence_vec = self._span_weighted_pool(
                last_4, span_logits, attention_mask.float(),
            )
            if CFG.use_cross_attention:
                mini_seq = torch.stack([promise_vec, evidence_vec], dim=1)
                attended = self.cross_attn(mini_seq)
                promise_vec = attended[:, 0, :]
                evidence_vec = attended[:, 1, :]
            parts.append(self.promise_proj(promise_vec))
            parts.append(self.evidence_proj(evidence_vec))

        combined = torch.cat(parts, dim=-1)

        # Cascade forward — 上游 detach 後送下游
        def _timeline_forward(head_input):
            """Ordinal 模式：回傳 5-class log_probs；標準模式：raw logits."""
            if self.use_ordinal_timeline:
                na_logit, ord_logits = self.task_heads["verification_timeline"](head_input)
                p5 = OrdinalTimelineHead.to_5class_probs(
                    na_logit, ord_logits, num_ord_classes=4,
                ).clamp(min=1e-8)
                return torch.log(p5)
            else:
                return self.task_heads["verification_timeline"](head_input)

        if getattr(CFG, "use_cascade_heads", False):
            promise_logits = self.task_heads["promise_status"](combined)
            promise_probs = F.softmax(promise_logits, dim=-1).detach()

            combined_p = torch.cat([combined, promise_probs], dim=-1)
            timeline_logits = _timeline_forward(combined_p)
            es_logits = self.task_heads["evidence_status"](combined_p)

            es_probs = F.softmax(es_logits, dim=-1).detach()
            combined_pe = torch.cat([combined, promise_probs, es_probs], dim=-1)
            quality_logits = self.task_heads["evidence_quality"](combined_pe)

            task_logits = {
                "promise_status":        promise_logits,
                "verification_timeline": timeline_logits,
                "evidence_status":       es_logits,
                "evidence_quality":      quality_logits,
            }
        else:
            task_logits = {}
            for f, head in self.task_heads.items():
                if f == "verification_timeline" and self.use_ordinal_timeline:
                    task_logits[f] = _timeline_forward(combined)
                else:
                    task_logits[f] = head(combined)

        return task_logits, span_logits


print("✅ MultiTaskRoberta（cascade 版本）已定義")


✅ MultiTaskRoberta（cascade 版本）已定義


In [5]:
# === Step 5 — Test Dataset ===

_ESG_NAMES = {"E": "環境", "S": "社會", "G": "治理"}


def build_esg_prefix(et: str) -> str:
    if not et:
        return ""
    parts = [t.strip() for t in et.split(";")]
    names = [_ESG_NAMES[t] for t in parts if t in _ESG_NAMES]
    return ("、".join(names) + "類：") if names else ""


class ESGTestDataset(Dataset):
    """推論用 Dataset：不含 label / span_labels，只產 input_ids / attention_mask / features."""

    def __init__(self, data, tokenizer):
        self.data = data
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        text = sample.get("data", "")
        # Test 集沒有 esg_type，build_esg_prefix 會 return ""，等於無前綴
        if CFG.use_esg_prefix:
            text = build_esg_prefix(sample.get("esg_type", "")) + text

        enc = self.tokenizer(
            text,
            truncation=True, max_length=CFG.max_len,
            padding="max_length", return_tensors="pt",
        )
        feats = (
            torch.tensor(extract_features(sample), dtype=torch.float)
            if CFG.use_hand_features
            else torch.zeros(CFG.num_hand_features, dtype=torch.float)
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "features": feats,
        }


print("✅ ESGTestDataset 已定義")


✅ ESGTestDataset 已定義


In [6]:
# === Step 6 — 載入 tokenizer + test 資料 + 跑 20-ckpt soft voting (4 seed × 5 fold) ===

print("📥 載入 tokenizer 與測試資料...")
tokenizer = AutoTokenizer.from_pretrained(CFG.model_name, use_fast=True)
test_data = json.load(open(TEST_JSON, encoding="utf-8"))
print(f"   Tokenizer: {CFG.model_name}")
print(f"   Test set: {len(test_data)} 筆（id {test_data[0]['id']} ~ {test_data[-1]['id']}）")

test_ds = ESGTestDataset(test_data, tokenizer)
# Windows + Jupyter 上 num_workers>0 會卡死（worker process 無法 pickle Jupyter context）
# 一律設 0 最穩，單 batch ~0.5-1s 差距可忽略。
test_loader = DataLoader(
    test_ds, batch_size=CFG.inference_batch_size, shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

print("\n📥 建立 inference 模型骨架（先載入原始 backbone，稍後會被 EMA 權重 overwrite）...")
model = MultiTaskRoberta(NUM_LABELS).to(device)
model.eval()
print("   ✅ 模型已就緒")


def _ckpt_label(ckpt_path: str) -> str:
    """從路徑取 fold 編號當進度條 desc，例如 'fold_1'。"""
    parts = ckpt_path.replace("\\", "/").split("/")
    return parts[-2]  # e.g. fold_1


@torch.no_grad()
def predict_one_ckpt(ckpt_path: str) -> Dict[str, torch.Tensor]:
    label = _ckpt_label(ckpt_path)
    print(f"\n  ▶ 載入 {label} ({ckpt_path}) ...")
    state = torch.load(ckpt_path, weights_only=False, map_location=device)
    ema = state.get("ema")
    if ema is None:
        raise RuntimeError(f"{ckpt_path} 沒有 'ema' key — 是否為非 EMA-only checkpoint？")

    incompat = model.load_state_dict(ema, strict=False)
    benign_tags = ("position_ids", "token_type_ids")
    bad_missing = [k for k in incompat.missing_keys if not any(t in k for t in benign_tags)]
    if bad_missing or incompat.unexpected_keys:
        raise RuntimeError(
            f"Checkpoint 與模型架構不匹配 ({ckpt_path}):\n"
            f"  missing: {bad_missing[:6]}\n"
            f"  unexpected: {list(incompat.unexpected_keys)[:6]}\n"
            f"  → 通常表示 CFG.num_hand_features 或 use_cascade_heads 設定不對"
        )
    print(f"    state_dict 載入完成，開始 forward...")

    probs_per_field = {f: [] for f in EVAL_FIELDS}
    pbar = tqdm(test_loader, desc=label, leave=True)
    for batch in pbar:
        ids = batch["input_ids"].to(device, non_blocking=True)
        mask = batch["attention_mask"].to(device, non_blocking=True)
        feats = batch["features"].to(device, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available(), dtype=torch.float16):
            logits, _ = model(ids, mask, feats)
        for f in EVAL_FIELDS:
            probs_per_field[f].append(F.softmax(logits[f].float(), dim=-1).cpu())
    return {f: torch.cat(v, dim=0) for f, v in probs_per_field.items()}


print(f"\n🔥 開始 {len(ckpt_paths)}-ckpt ensemble inference (4 seeds × 5 folds)...")
# 建立 ckpt → seed 反查表（供 per-seed 累加用）
_ckpt_to_seed = {}
for _seed, _paths in ckpt_paths_per_seed.items():
    for _p in _paths:
        _ckpt_to_seed[_p] = _seed

accum = {f: None for f in EVAL_FIELDS}                                   # 全體 uniform 用
seed_accum = {s: {f: None for f in EVAL_FIELDS} for s in CKPT_ZIPS}       # per-seed 累加器

for ckpt in ckpt_paths:
    p = predict_one_ckpt(ckpt)
    seed = _ckpt_to_seed[ckpt]
    for f in EVAL_FIELDS:
        accum[f] = p[f].clone() if accum[f] is None else accum[f] + p[f]
        seed_accum[seed][f] = (
            p[f].clone() if seed_accum[seed][f] is None
            else seed_accum[seed][f] + p[f]
        )

# Uniform averaging（全部 ckpt 等權重）
final_probs = {f: accum[f] / len(ckpt_paths) for f in EVAL_FIELDS}

# Per-seed averaging — 給 Step 7.B 產不同 ensemble 用
seed_avg_probs = {
    s: {f: seed_accum[s][f] / len(ckpt_paths_per_seed[s]) for f in EVAL_FIELDS}
    for s in CKPT_ZIPS
}

print(f"\n✅ {len(ckpt_paths)}-fold uniform ensemble inference 完成")
print(f"   final_probs (uniform {len(ckpt_paths)}-ckpt): {list(EVAL_FIELDS)}")
print(f"   seed_avg_probs (per-seed 5-fold): {list(seed_avg_probs.keys())}")
for f, p in final_probs.items():
    print(f"   {f:<22} shape={tuple(p.shape)}")


📥 載入 tokenizer 與測試資料...


   Tokenizer: hfl/chinese-roberta-wwm-ext
   Test set: 2000 筆（id 12001 ~ 14000）

📥 建立 inference 模型骨架（先載入原始 backbone，稍後會被 EMA 權重 overwrite）...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 20722.13it/s]
BertModel LOAD REPORT from: hfl/chinese-roberta-wwm-ext
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "c:\Users\genji\Downloads\tf-safe-project\.venv

   ✅ 模型已就緒

🔥 開始 20-ckpt ensemble inference (4 seeds × 5 folds)...

  ▶ 載入 fold_1 (checkpoints_inference_amask_1106910\fold_1\best.pt) ...
    state_dict 載入完成，開始 forward...


fold_1: 100%|██████████| 125/125 [00:14<00:00,  8.45it/s]


...


  ▶ 載入 fold_4 (checkpoints_inference_amask_9910\fold_4\best.pt) ...
    state_dict 載入完成，開始 forward...


fold_4: 100%|██████████| 125/125 [00:16<00:00,  7.58it/s]



  ▶ 載入 fold_5 (checkpoints_inference_amask_9910\fold_5\best.pt) ...
    state_dict 載入完成，開始 forward...


fold_5: 100%|██████████| 125/125 [00:16<00:00,  7.49it/s]


✅ 20-fold uniform ensemble inference 完成
   final_probs (uniform 20-ckpt): ['promise_status', 'verification_timeline', 'evidence_status', 'evidence_quality']
   seed_avg_probs (per-seed 5-fold): [1106910, 1106, 910, 9910]
   promise_status         shape=(2000, 2)
   verification_timeline  shape=(2000, 5)
   evidence_status        shape=(2000, 3)
   evidence_quality       shape=(2000, 4)


In [7]:
# === Step 6.5 — Per-seed OOF + Combined Ensemble OOF（15-ckpt 對比基準）===
# 每個 seed 用自己的 K-fold split 重建 → 拿沒訓過該樣本的 ckpt 預測 → 不偏
# 兩個 seed 各自報一份 OOF F1，再算「seed 間平均」的 ensemble OOF F1
try:
    from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
except ImportError:
    print("📥 安裝 iterative-stratification...")
    import subprocess, sys as _sys
    subprocess.check_call([_sys.executable, "-m", "pip", "install", "-q", "iterative-stratification"])
    from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

import numpy as np
from sklearn.metrics import f1_score


# ───── baseline 比較對象 ─────
PREV_BASELINE_OOF_F1 = 0.61636   # 舊 A+MASK seed=1106910 單獨 OOF
N_SPLITS             = 5
# ────────────────────────────


def _find_path(candidates):
    for p in candidates:
        if os.path.exists(p):
            return p
    return None


TRAIN_JSON_PATH = _find_path(["../vpesg4k_train_1000.json", "vpesg4k_train_1000.json"])
VAL_JSON_PATH   = _find_path(["../vpesg4k_val_1000.json",   "vpesg4k_val_1000.json"])
assert TRAIN_JSON_PATH and VAL_JSON_PATH, "找不到 train/val JSON"


# ── 重建 trainval（順序：train 在前、val 在後，與訓練一致）
_train = json.load(open(TRAIN_JSON_PATH, encoding="utf-8"))
_val   = json.load(open(VAL_JSON_PATH,   encoding="utf-8"))
for d in _val:
    if d.get("verification_timeline") == "more_than_5_years":
        d["verification_timeline"] = "longer_than_5_years"
trainval = _train + _val
print(f"📚 trainval: {len(trainval)} 筆")


# ── multilabel y matrix（兩 seed 共用）
def _build_multilabel_y(data):
    columns = [(f, lab) for f, labs in EVAL_FIELDS.items() for lab in labs]
    col_to_idx = {c: i for i, c in enumerate(columns)}
    y = np.zeros((len(data), len(columns)), dtype=np.int32)
    for i, d in enumerate(data):
        for f in EVAL_FIELDS:
            y[i, col_to_idx[(f, d[f])]] = 1
    return y


y_multilabel = _build_multilabel_y(trainval)


# ── 預測 helper
@torch.no_grad()
def _predict_on_loader(ckpt_path, loader):
    state = torch.load(ckpt_path, weights_only=False, map_location=device)
    model.load_state_dict(state["ema"], strict=False)
    out = {f: [] for f in EVAL_FIELDS}
    for batch in tqdm(loader, desc=_ckpt_label(ckpt_path), leave=False):
        ids = batch["input_ids"].to(device, non_blocking=True)
        mask = batch["attention_mask"].to(device, non_blocking=True)
        feats = batch["features"].to(device, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available(), dtype=torch.float16):
            logits, _ = model(ids, mask, feats)
        for f in EVAL_FIELDS:
            out[f].append(F.softmax(logits[f].float(), dim=-1).cpu())
    return {f: torch.cat(v, dim=0) for f, v in out.items()}


# ── 對每個 seed 各自跑 OOF
oof_probs_per_seed = {}   # seed -> {field: tensor [N, n_labels]}
for seed in CKPT_ZIPS.keys():
    print(f"\n━━━━━ Seed {seed} OOF inference ━━━━━")
    mskf = MultilabelStratifiedKFold(
        n_splits=N_SPLITS, shuffle=True, random_state=seed,
    )
    splits = list(mskf.split(np.arange(len(trainval)), y_multilabel))
    paths = ckpt_paths_per_seed[seed]
    oof = {f: torch.zeros((len(trainval), n), dtype=torch.float32)
           for f, n in NUM_LABELS.items()}
    for fold_idx, (_, va_idx) in enumerate(splits):
        va_loader = DataLoader(
            ESGTestDataset([trainval[i] for i in va_idx], tokenizer),
            batch_size=CFG.inference_batch_size, shuffle=False,
            num_workers=0, pin_memory=torch.cuda.is_available(),
        )
        print(f"   fold {fold_idx+1}/{N_SPLITS}: val={len(va_idx)} 筆")
        fold_probs = _predict_on_loader(paths[fold_idx], va_loader)
        for f in EVAL_FIELDS:
            oof[f][va_idx] = fold_probs[f]
    oof_probs_per_seed[seed] = oof
print(f"\n✅ 所有 seed OOF inference 完成")


# ── Decode helpers
def _decode_argmax(probs):
    n = next(iter(probs.values())).shape[0]
    return [
        {f: ID2LABEL[f][int(probs[f][i].argmax())] for f in EVAL_FIELDS}
        for i in range(n)
    ]


def _apply_constraints(preds):
    for p in preds:
        if p["promise_status"] == "No":
            p["verification_timeline"] = "N/A"
            p["evidence_status"]       = "N/A"
            p["evidence_quality"]      = "N/A"
        if p["evidence_status"] in ("No", "N/A"):
            p["evidence_quality"] = "N/A"
    return preds


_FIELD_WEIGHTS = {"promise_status": 0.2, "verification_timeline": 0.15,
                  "evidence_status": 0.3, "evidence_quality": 0.35}


def _eval_oof(oof, gt):
    """Decode + 邏輯約束 + per-task macro + weighted F1。"""
    preds = _apply_constraints(_decode_argmax(oof))
    per_task = {}
    weighted = 0.0
    for field, labels in EVAL_FIELDS.items():
        y_true = [g[field] for g in gt]
        y_pred = [p[field] for p in preds]
        macro = f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)
        per_task[field] = macro
        weighted += macro * _FIELD_WEIGHTS[field]
    return weighted, per_task


# ── (1) 各 seed 單獨 OOF
print(f"\n=========== 各 seed 單獨 OOF ===========")
per_seed_f1 = {}
for seed, oof in oof_probs_per_seed.items():
    weighted, per_task = _eval_oof(oof, trainval)
    per_seed_f1[seed] = weighted
    print(f"\nseed={seed}:")
    for field in EVAL_FIELDS:
        print(f"   {field:<22} macro={per_task[field]:.5f}")
    print(f"   weighted F1 = {weighted:.5f}")


# ── (2) 多 seed 平均的 ensemble OOF
# 對每筆樣本，平均「每個 seed 中沒訓到它的 ckpt 預測」
oof_ensemble = {
    f: torch.stack([oof_probs_per_seed[s][f] for s in CKPT_ZIPS.keys()]).mean(dim=0)
    for f in EVAL_FIELDS
}
ens_weighted, ens_per_task = _eval_oof(oof_ensemble, trainval)

print(f"\n=========== {len(CKPT_ZIPS)}-seed Ensemble OOF ===========")
for field in EVAL_FIELDS:
    print(f"   {field:<22} macro={ens_per_task[field]:.5f}")
print(f"   weighted F1 = {ens_weighted:.5f}")


# ── (3) 對照與判讀
print(f"\n=========== Summary ===========")
print(f"   舊版 A+MASK baseline      = {PREV_BASELINE_OOF_F1:.5f}")
for seed, f1 in per_seed_f1.items():
    delta = (f1 - PREV_BASELINE_OOF_F1) * 100
    print(f"   seed={seed:<8} OOF       = {f1:.5f}  ({delta:+.3f}pp vs baseline)")
ens_delta = (ens_weighted - PREV_BASELINE_OOF_F1) * 100
best_single = max(per_seed_f1.values())
gain_over_best_single = (ens_weighted - best_single) * 100
print(f"   ★ {len(CKPT_ZIPS)}-seed Ensemble OOF = {ens_weighted:.5f}  ({ens_delta:+.3f}pp vs baseline)")
print(f"                                          ({gain_over_best_single:+.3f}pp vs best single seed)")

if ens_delta >= 0.3:
    print(f"\n⭐ Multi-seed 有實質增益 — 直接提交 {len(ckpt_paths)}-ckpt 版本")
elif ens_delta >= 0.1:
    print(f"\n🟡 微增益 — 可能在 noise 邊緣，提交看看")
elif ens_delta >= -0.1:
    print(f"\n⚪ 等同 baseline — multi-seed 沒幫助")
else:
    print(f"\n🔴 退步 — 不正常，檢查兩個 seed 的 ckpt 是否正確")


📚 trainval: 2000 筆

━━━━━ Seed 1106910 OOF inference ━━━━━
   fold 1/5: val=400 筆


   fold 2/5: val=400 筆


   fold 3/5: val=398 筆


   fold 4/5: val=401 筆


   fold 5/5: val=401 筆


...

   fold 2/5: val=399 筆


   fold 3/5: val=400 筆


   fold 4/5: val=400 筆


   fold 5/5: val=400 筆



✅ 所有 seed OOF inference 完成

=========== 各 seed 單獨 OOF ===========

seed=1106910:
   promise_status         macro=0.80974
   verification_timeline  macro=0.57550
   evidence_status        macro=0.70115
   evidence_quality       macro=0.45069
   weighted F1 = 0.61636

seed=1106:
   promise_status         macro=0.79719
   verification_timeline  macro=0.58137
   evidence_status        macro=0.68888
   evidence_quality       macro=0.46402
   weighted F1 = 0.61571

seed=910:
   promise_status         macro=0.80372
   verification_timeline  macro=0.59660
   evidence_status        macro=0.69220
   evidence_quality       macro=0.45472
   weighted F1 = 0.61705

seed=9910:
   promise_status         macro=0.80757
   verification_timeline  macro=0.58318
   evidence_status        macro=0.70390
   evidence_quality       macro=0.46745
   weighted F1 = 0.62377

=========== 4-seed Ensemble OOF ===========
   promise_status         macro=0.81466
   verification_timeline  macro=0.59144
   evidence_status

In [8]:
# === Step 7 — Decode test + Write CSV（Uniform 15-ckpt ensemble）===
# Per-task selection 已撤（2026-06-16 實證 OOF 看似強但 LB 反而 −0.28pp，overfit）
# 維持單純 15-ckpt uniform soft voting — 已驗證 LB 0.6206 為目前最佳

# Post-processing 已驗證 alpha=0 / thr=0 為最佳
PRIOR_CORRECTION_ALPHA = 0.0
CONF_THRESHOLD         = 0.0
log_priors             = None

probs_for_decode = final_probs
MODE_TAG = f"uniform_{len(ckpt_paths)}ckpt"
print(f"📌 Decode 模式: Uniform {len(ckpt_paths)}-ckpt ensemble")


# Decode + 邏輯約束 functions（與 Step 6.5 同邏輯，這裡公開命名）
def decode(probs, log_priors=None, alpha=0.0):
    n = next(iter(probs.values())).shape[0]
    use_corr = log_priors is not None and alpha > 0
    out = []
    for i in range(n):
        p = {}
        for f in EVAL_FIELDS:
            scores = torch.log(probs[f][i].clamp(min=1e-9))
            if use_corr:
                scores = scores - alpha * log_priors[f]
            p[f] = ID2LABEL[f][int(scores.argmax())]
        out.append(p)
    return out


def apply_logic_constraints(preds, probs=None, conf_threshold=0.0):
    use_conf = probs is not None and conf_threshold > 0
    if use_conf:
        eq_clear_id = LABEL2ID["evidence_quality"]["Clear"]
        es_yes_id = LABEL2ID["evidence_status"]["Yes"]
    for i, p in enumerate(preds):
        if p["promise_status"] == "No":
            p["verification_timeline"] = "N/A"
            p["evidence_status"] = "N/A"
            p["evidence_quality"] = "N/A"
        if p["evidence_status"] in ("No", "N/A"):
            p["evidence_quality"] = "N/A"
        if use_conf and p["evidence_quality"] == "Clear":
            eq_conf = float(probs["evidence_quality"][i, eq_clear_id])
            es_conf = float(probs["evidence_status"][i, es_yes_id])
            if eq_conf < conf_threshold and es_conf < conf_threshold:
                p["evidence_quality"] = "Not Clear"
    return preds


def to_submission_label(field: str, label: str) -> str:
    if field == "verification_timeline" and label == "longer_than_5_years":
        return "more_than_5_years"
    return label


print(f"\n=== 套用於 test ===")
print(f"   PRIOR_CORRECTION_ALPHA = {PRIOR_CORRECTION_ALPHA}")
print(f"   CONF_THRESHOLD         = {CONF_THRESHOLD}")

preds = decode(probs_for_decode, log_priors=log_priors, alpha=PRIOR_CORRECTION_ALPHA)
preds = apply_logic_constraints(preds, probs=probs_for_decode, conf_threshold=CONF_THRESHOLD)
print(f"\n✅ Decode + 邏輯約束完成（{len(preds)} 筆）\n")

# 分布診斷
for field in EVAL_FIELDS:
    c = Counter(p[field] for p in preds)
    print(f"{field}:")
    for k, v in sorted(c.items(), key=lambda x: -x[1]):
        out_k = to_submission_label(field, k)
        print(f"  {out_k:<26} {v:>4}")
    print()

# 寫 CSV（main + archive 兩份）
ARCHIVE_CSV = f"submission_{MODE_TAG}.csv"

FIELDS_ORDER = ["id", "promise_status", "verification_timeline",
                "evidence_status", "evidence_quality"]
for out_path in [OUTPUT_CSV, ARCHIVE_CSV]:
    with open(out_path, "w", encoding="utf-8", newline="") as f:
        w = csv.writer(f)
        w.writerow(FIELDS_ORDER)
        for sample, pred in zip(test_data, preds):
            w.writerow([
                sample["id"],
                pred["promise_status"],
                to_submission_label("verification_timeline", pred["verification_timeline"]),
                pred["evidence_status"],
                pred["evidence_quality"],
            ])

print(f"✅ 寫入 {OUTPUT_CSV} ({os.path.getsize(OUTPUT_CSV)} bytes)")
print(f"✅ 寫入 {ARCHIVE_CSV} ({os.path.getsize(ARCHIVE_CSV)} bytes)（archive 備用）")


📌 Decode 模式: Uniform 20-ckpt ensemble

=== 套用於 test ===
   PRIOR_CORRECTION_ALPHA = 0.0
   CONF_THRESHOLD         = 0.0

✅ Decode + 邏輯約束完成（2000 筆）

promise_status:
  Yes                        1697
  No                          303

verification_timeline:
  already                     746
  between_2_and_5_years       557
  more_than_5_years           372
  N/A                         303
  within_2_years               22

evidence_status:
  Yes                        1443
  N/A                         303
  No                          254

evidence_quality:
  Clear                      1231
  N/A                         557
  Not Clear                   212

✅ 寫入 submission.csv (67711 bytes)
✅ 寫入 submission_uniform_20ckpt.csv (67711 bytes)（archive 備用）


In [9]:
# === Step 7.B — 產 9910-only 與 3-best 兩份提交 CSV ===
# (A) seed=9910 alone (5 ckpt)
# (B) 3-best (1106910 + 910 + 9910, 15 ckpt — drop seed 1106)
# 跑完得到 submission_seed9910_only.csv 跟 submission_3best_no1106.csv

import csv as _csv
from collections import Counter as _Counter


def _build_preds_from_probs(probs):
    """跟 Step 7 同邏輯：decode + apply logic constraints。"""
    p_decoded = decode(probs, log_priors=None, alpha=0.0)
    p_decoded = apply_logic_constraints(p_decoded, probs=probs, conf_threshold=0.0)
    return p_decoded


def _write_one_csv(preds, output_path, tag):
    with open(output_path, "w", encoding="utf-8", newline="") as f:
        w = _csv.writer(f)
        w.writerow(FIELDS_ORDER)
        for sample, pred in zip(test_data, preds):
            w.writerow([
                sample["id"],
                pred["promise_status"],
                to_submission_label("verification_timeline", pred["verification_timeline"]),
                pred["evidence_status"],
                pred["evidence_quality"],
            ])
    print(f"   ✅ 寫入 {output_path} ({os.path.getsize(output_path)} bytes)")

    # Distribution diagnostic
    print(f"\n   {tag} 各 task 分布:")
    for field in EVAL_FIELDS:
        c = _Counter(p[field] for p in preds)
        line = "  ".join(
            f"{to_submission_label(field, k)}:{v}"
            for k, v in sorted(c.items(), key=lambda x: -x[1])
        )
        print(f"     {field:<22}: {line}")


# ─── (A) seed=9910 alone ───
print("=" * 60)
print("📝 (A) seed=9910 alone (5 ckpt)")
print("=" * 60)
assert 9910 in seed_avg_probs, "Step 6 沒抽到 seed=9910 的 per-seed probs"
probs_a = seed_avg_probs[9910]
preds_a = _build_preds_from_probs(probs_a)
_write_one_csv(preds_a, "submission_seed9910_only.csv", "[9910 alone]")


# ─── (B) 3-best (1106910 + 910 + 9910) ───
print("\n" + "=" * 60)
print("📝 (B) 3-best ensemble (1106910 + 910 + 9910, 15 ckpt)")
print("=" * 60)
SEEDS_3BEST = [1106910, 910, 9910]
for s in SEEDS_3BEST:
    assert s in seed_avg_probs, f"Step 6 沒抽到 seed={s}"

probs_b = {
    f: sum(seed_avg_probs[s][f] for s in SEEDS_3BEST) / len(SEEDS_3BEST)
    for f in EVAL_FIELDS
}
preds_b = _build_preds_from_probs(probs_b)
_write_one_csv(preds_b, "submission_3best_no1106.csv", "[3-best]")


# ─── (C) 2-best (910 + 9910，OOF 兩強 seed) ───
print("\n" + "=" * 60)
print("📝 (C) 2-best ensemble (910 + 9910, 10 ckpt)")
print("=" * 60)
SEEDS_2BEST = [910, 9910]
for s in SEEDS_2BEST:
    assert s in seed_avg_probs, f"Step 6 沒抽到 seed={s}"

probs_c = {
    f: sum(seed_avg_probs[s][f] for s in SEEDS_2BEST) / len(SEEDS_2BEST)
    for f in EVAL_FIELDS
}
preds_c = _build_preds_from_probs(probs_c)
_write_one_csv(preds_c, "submission_2best_910_9910.csv", "[2-best]")


# ─── Summary ───
print("\n" + "=" * 60)
print("📤 三份 alternative CSV 已產出：")
print("   submission_seed9910_only.csv     ← (A) seed=9910 only (5 ckpt)")
print("   submission_3best_no1106.csv      ← (B) 3-best 1106910+910+9910 (15 ckpt)")
print("   submission_2best_910_9910.csv    ← (C) 2-best 910+9910 (10 ckpt)  ← 新")
print("\n額外存在的 submission_uniform_20ckpt.csv (4-seed full) 不要交")
print("\n已知 LB:")
print("   (A) 9910 alone:  0.6162")
print("   (B) 3-best:      0.6243 ⭐")
print("   (C) 2-best:      未知（這次新交）")


📝 (A) seed=9910 alone (5 ckpt)
   ✅ 寫入 submission_seed9910_only.csv (67609 bytes)

   [9910 alone] 各 task 分布:
     promise_status        : Yes:1666  No:334
     verification_timeline : already:707  between_2_and_5_years:575  more_than_5_years:361  N/A:334  within_2_years:23
     evidence_status       : Yes:1398  N/A:334  No:268
     evidence_quality      : Clear:1184  N/A:602  Not Clear:214

📝 (B) 3-best ensemble (1106910 + 910 + 9910, 15 ckpt)
   ✅ 寫入 submission_3best_no1106.csv (67754 bytes)

   [3-best] 各 task 分布:
     promise_status        : Yes:1699  No:301
     verification_timeline : already:747  between_2_and_5_years:559  more_than_5_years:372  N/A:301  within_2_years:21
     evidence_status       : Yes:1445  N/A:301  No:254
     evidence_quality      : Clear:1231  N/A:555  Not Clear:214

📝 (C) 2-best ensemble (910 + 9910, 10 ckpt)
   ✅ 寫入 submission_2best_910_9910.csv (67661 bytes)

   [2-best] 各 task 分布:
     promise_status        : Yes:1687  No:313
     verification_timeline

In [10]:
# === Step 7.5 — Misleading Injection 啟發式分析（針對 test 上 Misleading=0 的問題）===
# 你目前 test 的 evidence_quality 預測中 Misleading=0 → 該類 macro F1 必為 0
# 此 cell 用「啟發式 score」找出最可能被誤判為 Not Clear 的 Misleading 樣本
# 顯示 ranked list，由你決定要 flip 幾個（保守為上）

# 評分組件:
#   1. ML_prob_rank: Misleading 機率在 4 個 EQ 類別中的排名（越前越可疑）
#   2. ML_vs_argmax_gap: argmax(Not Clear) 機率 - Misleading 機率（越小越可疑）
#   3. strategic_vague_keyword 密度: 含「卓越/主軸/中長期/碳策略」等 greenwashing 詞彙
#   4. long_term_goal 密度:「2030/2050/淨零/科學基礎」等遠期 vague 目標
#   5. promise=Yes 且 evidence_status=Yes 但 Not Clear 的 EQ 信心不高 (≤0.6)

import re as _re

NOT_CLEAR_ID  = LABEL2ID["evidence_quality"]["Not Clear"]
MISLEADING_ID = LABEL2ID["evidence_quality"]["Misleading"]
CLEAR_ID      = LABEL2ID["evidence_quality"]["Clear"]
NA_ID         = LABEL2ID["evidence_quality"]["N/A"]

# 已含於 Step 3 的 PATTERNS dict：strategic_vague / long_term_goal
_STRATEGIC_VAGUE = PATTERNS.get("strategic_vague",
    _re.compile(r"卓越|主軸|共創|中長期|碳策略|核心|藍圖|優化|期望"))
_LONG_TERM = PATTERNS.get("long_term_goal",
    _re.compile(r"2030|2050|2040|淨零|科學基礎|減量目標|減量路徑|脫碳|再生能源"))


def score_misleading_candidate(i, preds, eq_probs, sample):
    """對 test sample i 算「該被 flip 成 Misleading」的可能分數 (0~1)。

    僅評估「目前 EQ 預測為 Not Clear」的樣本（最容易語意上跟 Misleading 混淆）。
    """
    if preds[i]["evidence_quality"] != "Not Clear":
        return None    # 跳過：不是 Not Clear，不在候選範圍

    probs = eq_probs[i]
    argmax_idx = int(probs.argmax())
    if argmax_idx != NOT_CLEAR_ID:
        return None    # 雙重保險

    ml_prob = float(probs[MISLEADING_ID])
    nc_prob = float(probs[NOT_CLEAR_ID])
    clear_prob = float(probs[CLEAR_ID])

    # Component 1: ML rank（4 個類別裡 Misleading 排第幾）
    sorted_probs = sorted(enumerate(probs.tolist()), key=lambda x: -x[1])
    ml_rank = next(r for r, (idx, _) in enumerate(sorted_probs) if idx == MISLEADING_ID)
    rank_score = (3 - ml_rank) / 3.0      # rank 0 → 1.0, rank 3 → 0.0

    # Component 2: gap between Not Clear (argmax) and Misleading（越小越可疑）
    gap = nc_prob - ml_prob
    gap_score = max(0.0, 1.0 - gap * 3.0)  # gap=0.33 → 0, gap=0 → 1

    # Component 3-4: keyword density
    text = sample.get("data", "")
    sv = len(_STRATEGIC_VAGUE.findall(text))
    lt = len(_LONG_TERM.findall(text))
    keyword_score = min(1.0, (sv * 2 + lt) / 6.0)

    # Component 5: low confidence Not Clear（NC argmax 但信心不夠）
    low_conf = 1.0 if nc_prob < 0.55 else max(0, 1.0 - (nc_prob - 0.55) * 4)

    # Composite (各 component 等權，可微調)
    total = (0.25 * rank_score + 0.25 * gap_score
             + 0.25 * keyword_score + 0.25 * low_conf)
    return {
        "score":      total,
        "ml_prob":    ml_prob,
        "nc_prob":    nc_prob,
        "clear_prob": clear_prob,
        "gap":        gap,
        "ml_rank":    ml_rank,
        "sv_count":   sv,
        "lt_count":   lt,
    }


# ── 算分數
print("📐 分析 Misleading 候選樣本...")
eq_probs = probs_for_decode["evidence_quality"]
candidates = []
for i in range(len(preds)):
    score_info = score_misleading_candidate(i, preds, eq_probs, test_data[i])
    if score_info is not None:
        candidates.append((i, score_info))

candidates.sort(key=lambda x: -x[1]["score"])
print(f"   候選樣本 (current EQ=Not Clear): {len(candidates)}")


# ── 顯示 Top-K
TOP_K_DISPLAY = 25
print(f"\n=== Top {TOP_K_DISPLAY} Misleading injection candidates ===")
print(f"{'rank':>4} {'id':>6} {'score':>6} {'ml_p':>6} {'nc_p':>6} {'gap':>6} "
      f"{'sv':>3} {'lt':>3}  text snippet")
print("-" * 110)
for rank, (idx, info) in enumerate(candidates[:TOP_K_DISPLAY]):
    text = test_data[idx].get("data", "")[:50].replace("\n", " ")
    print(f"{rank+1:>4} {test_data[idx]['id']:>6} {info['score']:>6.3f} "
          f"{info['ml_prob']:>6.3f} {info['nc_prob']:>6.3f} {info['gap']:>6.3f} "
          f"{info['sv_count']:>3} {info['lt_count']:>3}  {text}...")


# ── Flip 控制
# 設 0 = 不 flip（純分析）；設 N = flip top-N 個成 Misleading
# 建議從小開始（N=3~5），看 leaderboard 反應再調整
# 若 test 真的有 Misleading 樣本但模型完全沒抓到，每對一個都拉 EQ macro
NUM_TO_FLIP = 0      # ← 改這個數字啟用注入

if NUM_TO_FLIP > 0:
    print(f"\n🔄 Flipping top {NUM_TO_FLIP} candidates 成 Misleading...")
    flipped_ids = []
    for idx, info in candidates[:NUM_TO_FLIP]:
        old = preds[idx]["evidence_quality"]
        preds[idx]["evidence_quality"] = "Misleading"
        flipped_ids.append(test_data[idx]["id"])
        print(f"   id={test_data[idx]['id']}: {old} → Misleading (score={info['score']:.3f})")

    # 寫一份注入版 CSV
    INJECTED_CSV = f"submission_{MODE_TAG}_mlflip{NUM_TO_FLIP}.csv"
    with open(INJECTED_CSV, "w", encoding="utf-8", newline="") as f:
        w = csv.writer(f)
        w.writerow(FIELDS_ORDER)
        for sample, pred in zip(test_data, preds):
            w.writerow([
                sample["id"],
                pred["promise_status"],
                to_submission_label("verification_timeline", pred["verification_timeline"]),
                pred["evidence_status"],
                pred["evidence_quality"],
            ])
    print(f"\n✅ 寫入 {INJECTED_CSV}")
    print(f"   Flipped ids: {flipped_ids}")

    # 更新分布
    new_eq_dist = Counter(p["evidence_quality"] for p in preds)
    print(f"\n   新 EQ 分布: {dict(new_eq_dist)}")
else:
    print(f"\n📌 NUM_TO_FLIP=0 → 純分析，未寫 injected CSV")
    print(f"   要啟用：改 NUM_TO_FLIP=3~10（保守起見 ≤10）")


📐 分析 Misleading 候選樣本...
   候選樣本 (current EQ=Not Clear): 212

=== Top 25 Misleading injection candidates ===
rank     id  score   ml_p   nc_p    gap  sv  lt  text snippet
--------------------------------------------------------------------------------------------------------------
   1  13587  0.583  0.003  0.529  0.526   3   1  聯電致力整合營運策略與永續發展，自晶圓製造之核心業務出發，聚焦 10 項聯合國永續發展目標 (Sus...
   2  12367  0.542  0.005  0.509  0.504   0   5  不僅於此，為達成 2030 年聯發科技集團辦公室全面使用再生能源，以及 2050 年淨零排放目標，我...
   3  12588  0.523  0.003  0.610  0.607   1   5  智邦預訂 2030 年減少碳排 50%，透過內部的製程優化與產品設計，同時與供應鏈夥伴一起推動零碳鏈...
   4  12713  0.485  0.006  0.606  0.600   0   5  本公司配合持續增長的產能、合理的減碳規劃、PCB 業界趨勢，以及地區與國際規範等因子，已設定以 20...
   5  12235  0.417  0.002  0.514  0.512   1   0  統一企業關注智慧化生產以提升產品製作效率，本公司向經濟部申請測試計畫，測試茶飲產線的智慧生產，透過智...
   6  12382  0.417  0.006  0.530  0.524   1   0  因應2024年度風險評估結果，廣達識別出涵蓋人才永續、碳排管理、供應鏈治理與內部管理機制等八項高風險...
   7  12624  0.417  0.003  0.513  0.510   1   0  持續打造符合低耗能要求的 PCBA 生產線 (如節能型迴焊爐架設、貼片機節能模式優化)，以降低製造過..

In [11]:
# === Step 8 — Sanity Check：確認 submission.csv 符合規範 ===
import csv

with open(OUTPUT_CSV, encoding="utf-8", newline="") as f:
    rows = list(csv.reader(f))

print(f"總行數: {len(rows)} (header + {len(rows)-1} 資料列)")
assert len(rows) == 2001, f"預期 2001 行 (header + 2000 筆)，實際 {len(rows)}"

header = rows[0]
expected_header = ["id", "promise_status", "verification_timeline",
                   "evidence_status", "evidence_quality"]
assert header == expected_header, f"Header 不對:\n  預期: {expected_header}\n  實際: {header}"

# 檢查 id 順序 12001 ~ 14000
ids = [int(r[0]) for r in rows[1:]]
assert ids[0] == 12001, f"首 id 應為 12001，實際 {ids[0]}"
assert ids[-1] == 14000, f"末 id 應為 14000，實際 {ids[-1]}"
assert ids == list(range(12001, 14001)), "id 順序錯誤或有跳號"

# 檢查每個欄位的值都是合法 label
VALID = {
    "promise_status": {"Yes", "No"},
    "verification_timeline": {"already", "within_2_years", "between_2_and_5_years",
                              "more_than_5_years", "N/A"},
    "evidence_status": {"Yes", "No", "N/A"},
    "evidence_quality": {"Clear", "Not Clear", "Misleading", "N/A"},
}
for i, row in enumerate(rows[1:], start=1):
    for col_idx, field in enumerate(["promise_status", "verification_timeline",
                                       "evidence_status", "evidence_quality"], start=1):
        v = row[col_idx]
        assert v in VALID[field], f"row {i} {field}={v!r} 非合法 label"

# 檢查邏輯規則
for i, row in enumerate(rows[1:], start=1):
    _id, ps, vt, es, eq = row
    if ps == "No":
        assert vt == "N/A" and es == "N/A" and eq == "N/A", \
            f"row {i}: promise=No 但下游不全 N/A"
    if es in ("No", "N/A"):
        assert eq == "N/A", f"row {i}: evidence_status={es} 但 evidence_quality={eq}"

print("\n✅ Sanity check 全部通過：")
print(f"   - 2000 筆資料、id 12001-14000 順序正確")
print(f"   - 欄位順序、合法 label 全對")
print(f"   - 邏輯規則無衝突")
print(f"\n📤 {OUTPUT_CSV} 可直接提交給 AICup")

# 顯示前 5 筆
print("\n=== 前 5 筆 ===")
for r in rows[:6]:
    print("  " + ",".join(r))


總行數: 2001 (header + 2000 資料列)

✅ Sanity check 全部通過：
   - 2000 筆資料、id 12001-14000 順序正確
   - 欄位順序、合法 label 全對
   - 邏輯規則無衝突

📤 submission.csv 可直接提交給 AICup

=== 前 5 筆 ===
  id,promise_status,verification_timeline,evidence_status,evidence_quality
  12001,Yes,more_than_5_years,Yes,Clear
  12002,Yes,already,Yes,Clear
  12003,Yes,already,Yes,Clear
  12004,Yes,between_2_and_5_years,Yes,Clear
  12005,Yes,already,Yes,Clear


## 提交流程

1. 跑完所有 cell，會產出 `submission.csv` 在當前目錄
2. 上傳到 AICup 競賽系統

## 如果失敗了

| 錯誤訊息 | 可能原因 | 解法 |
|---|---|---|
| `Checkpoint 與模型架構不匹配` | `CFG.num_hand_features` 不對（17 vs 20）或某個 architecture flag 不一致 | 確認 CFG 完全等於訓練時 |
| `CUDA out of memory` | `inference_batch_size=16` 太大 | 改成 8 或 4 |
| `KeyError: 'ema'` | Checkpoint 不是 EMA-only 格式 | 看 Step 5.5 剝離流程 |
| `Sanity check 失敗` | 某 row label 非法 / 邏輯衝突 | 通常是 logic_constraints bug，重檢查 |
